In [1]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

# Load the raw datasets
agents = pd.read_csv("agents.csv")
transactions = pd.read_csv("agent_transactions.csv")

# Run the cleaning pipeline
clean_df, audit_log = clean_agent_data(transactions, agents)

# Display the results
print("Original transactions:", len(transactions))
print("Cleaned transactions:", len(clean_df))
print("Number of agents:", len(agents))

display(clean_df.head())

Original transactions: 612
Cleaned transactions: 600
Number of agents: 20


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,agent_match_status
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,Matched
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,Matched
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,Matched
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,Matched
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,Matched


In [3]:
# Display the data cleaning audit log
display(audit_log)

,issue,dimension,column,rows_affected
0,Duplicate transactions removed,Uniqueness,txn_id,12
1,Transaction types standardized,Consistency,txn_type,376
2,Invalid amounts converted to missing values,Validity,amount,0
3,Missing amounts imputed using median,Completeness,amount,20
4,Potential amount outliers flagged,Accuracy,amount,53
5,Transaction dates standardized,Consistency,txn_date,5
6,Transactions with unknown agents identified,Referential Integrity,agent_id,7


In [4]:
# Validate the cleaned dataset after merging agent details

print("POST-MERGE VALIDATION")
print("---------------------")

print("Cleaned transaction rows:", len(clean_df))
print("Unique transaction IDs:", clean_df["txn_id"].nunique())

print(
    "Unknown agents:",
    (clean_df["agent_match_status"] == "Unknown agent").sum()
)

print(
    "Matched transactions:",
    (clean_df["agent_match_status"] == "Matched").sum()
)

print("\nCounty information:")
print("Missing county values:", clean_df["county"].isna().sum())

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts())

POST-MERGE VALIDATION
---------------------
Cleaned transaction rows: 600
Unique transaction IDs: 600
Unknown agents: 7
Matched transactions: 593

County information:
Missing county values: 7

Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: Int64


In [5]:
# Summarize agent transactions by county

county_summary = (
    clean_df[clean_df["county"].notna()]
    .groupby("county")
    .agg(
        total_transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        unique_agents=("agent_id", "nunique")
    )
    .reset_index()
    .sort_values("total_amount", ascending=False)
)

display(county_summary.round(2))

,county,total_transactions,total_amount,average_amount,unique_agents
3,Nairobi,180,705626.0,3920.14,6
2,Mombasa,133,517578.0,3891.56,5
4,Nakuru,99,513484.5,5186.71,3
1,Kisumu,111,512795.0,4619.77,4
0,Kiambu,70,411787.5,5882.68,2


In [7]:
# Prepare transaction dates
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])

# Group transactions by week
weekly_summary = (
    clean_df
    .set_index("txn_date")
    .resample("W")
    .agg(
        total_transactions=("txn_id", "count"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

# Display weekly results without datetime rounding warnings
weekly_summary["total_amount"] = weekly_summary["total_amount"].round(2)

display(weekly_summary.style.format({
    "total_amount": "{:,.2f}"
}))

,txn_date,total_transactions,total_amount
0,2026-03-01 00:00:00,8,"18,441.50"
1,2026-03-08 00:00:00,74,"717,460.50"
2,2026-03-15 00:00:00,66,"403,025.50"
3,2026-03-22 00:00:00,65,"159,865.00"
4,2026-03-29 00:00:00,58,"116,003.00"
5,2026-04-05 00:00:00,67,"411,613.00"
6,2026-04-12 00:00:00,64,"420,683.00"
7,2026-04-19 00:00:00,73,"150,996.00"
8,2026-04-26 00:00:00,87,"176,559.00"
9,2026-05-03 00:00:00,38,"113,780.50"



# DATA QUALITY REPORT

## 1. Completeness
The original dataset contained 22 missing transaction amounts.
After removing 12 duplicate records, 20 missing amounts remained.
These were imputed using the median transaction amount.

## 2. Accuracy
Potential transaction amount outliers were identified using the
Interquartile Range (IQR) method. A total of 53 records were flagged
for further review rather than automatically deleted.

## 3. Consistency
Transaction types were standardized into Deposit and Withdrawal.
Five transaction dates were converted into the standard ISO date format.

## 4. Uniqueness
The original dataset contained 12 duplicate transaction IDs.
These duplicates were removed, leaving 600 unique transactions.

## 5. Validity
Transaction amounts were converted into numeric values.
The cleaning pipeline also checks date validity and standardizes
transaction categories.

## 6. Referential Integrity
Transactions were matched with the agent master dataset.
A total of 593 transactions were matched successfully, while
7 transactions contained unknown agent IDs. These records were
retained and clearly identified.

## Overall Conclusion
The cleaning process transformed 612 raw transaction records into
600 unique and standardized records. Missing amounts were imputed,
potential outliers were flagged, and agent information was merged.
The resulting dataset is suitable for further exploratory analysis,
subject to review of flagged outliers and unmatched agent records.
